# Spotify Music Recommender System

**Project:** Advanced Data Engineering — Content-Based Recommender Engine  
**Authors:** Pau Rossell & David Redrejo  

This notebook demonstrates all four data archetypes from the Technical Design Report:  
1. **Structured Matrices** — Dense audio feature extraction  
2. **Sparse Architectures** — Genre/Artist encoding with `sklearn`  
3. **Unstructured Data** — Album art color histograms  
4. **Graph Modeling** — Artist collaboration network  

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.impute import SimpleImputer

plt.style.use('ggplot')
print('Libraries loaded.')

---
## ARCHETYPE 1: Structured Matrix of Observations

Each row is a track (entity). Each column is an audio feature (attribute).  
This is the foundational $\text{Observations} \times \text{Variables}$ matrix.

In [ ]:
# Load dataset
df = pd.read_csv('../data/raw/dataset.csv', index_col=0)
print(f'Matrix shape: {df.shape[0]} Observations × {df.shape[1]} Variables')
df.head(3)

In [ ]:
# --- Data Integrity: Missingness Analysis (MCAR / MAR / MNAR) ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])

# Tracks without an identity (track_name, artists) cannot be recommended → Listwise Deletion (MCAR)
df = df.dropna(subset=['track_name', 'artists']).reset_index(drop=True)
print(f'\nShape after identity Listwise Deletion: {df.shape}')

In [ ]:
# --- Numerical Audio Feature Matrix ---
AUDIO_FEATURES = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

X_audio_raw = df[AUDIO_FEATURES].copy()

# Numerical missingness → Median imputation (MAR assumption)
imputer = SimpleImputer(strategy='median')
X_audio_imputed = imputer.fit_transform(X_audio_raw)

# Normalize: prevent high-variance features (e.g. tempo ~120) from dominating low-variance ones (e.g. acousticness ~0.3)
scaler = StandardScaler()
X_audio_scaled = scaler.fit_transform(X_audio_imputed)

print(f'Dense Audio Matrix: {X_audio_scaled.shape}')
print(f'Memory (dense): {X_audio_scaled.nbytes / 1024:.1f} KB')

In [ ]:
# --- EDA: Distribution of key audio features ---
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features_to_plot = ['danceability', 'energy', 'valence', 'tempo',
                    'acousticness', 'instrumentalness', 'speechiness', 'loudness']
for ax, col in zip(axes.flatten(), features_to_plot):
    df[col].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Audio Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ARCHETYPE 2: Sparse Matrix Architectures (CSR)

One-hot encoding `track_genre` and `artists` across 114k tracks produces a **very sparse** matrix.  
We use `TfidfVectorizer` (from scikit-learn) which internally returns a **CSR sparse matrix** — no dense conversion needed.

In [ ]:
# --- Genre Sparse Matrix ---
# Each row = a track's genre. TF-IDF weights frequent genres less (better discriminator)
genre_vectorizer = TfidfVectorizer()
X_genre = genre_vectorizer.fit_transform(df['track_genre'].fillna(''))

print(f'Genre CSR Matrix : {X_genre.shape} | Format: {X_genre.format}')
print(f'  Sparse memory  : {X_genre.data.nbytes / 1024:.1f} KB')
print(f'  Dense equiv.   : {(X_genre.shape[0] * X_genre.shape[1] * 8) / 1024:.1f} KB')

In [ ]:
# --- Artist Sparse Matrix ---
# Artists are semicolon-separated (e.g., "Brandi Carlile;Sam Smith")
# Replace ';' with ' ' so TfidfVectorizer treats each artist as a separate token
artist_text = df['artists'].fillna('').str.replace(';', ' ')
artist_vectorizer = TfidfVectorizer()
X_artists = artist_vectorizer.fit_transform(artist_text)

print(f'Artist CSR Matrix: {X_artists.shape} | Format: {X_artists.format}')
print(f'  Sparse memory  : {X_artists.data.nbytes / 1024:.1f} KB')

In [ ]:
# --- Combine all Features (hstack) ---
# Audio matrix must be converted to sparse to stack with genre/artist CSR matrices
X_audio_sparse = sp.csr_matrix(X_audio_scaled)

# Horizontal stack: all archetypes combined into a single feature matrix
X_combined = sp.hstack([X_audio_sparse, X_genre, X_artists], format='csr')

print(f'Combined Feature Matrix: {X_combined.shape}')
print(f'Total sparse memory    : {X_combined.data.nbytes / 1024 / 1024:.2f} MB')

---
## ARCHETYPE 4: Graph Modeling — Artist Collaboration Network

We parse multi-artist tracks to build a relational graph $G = (V, E)$.  
Each node is an artist; each edge represents a collaboration.

In [ ]:
import networkx as nx
from itertools import combinations

# Build edge list from multi-artist tracks
G = nx.Graph()

for _, row in df.iterrows():
    artists = [a.strip() for a in str(row['artists']).split(';')]
    if len(artists) > 1:
        for a1, a2 in combinations(artists, 2):
            if G.has_edge(a1, a2):
                G[a1][a2]['weight'] += 1
            else:
                G.add_edge(a1, a2, weight=1)

print(f'Graph G = ({G.number_of_nodes()} vertices, {G.number_of_edges()} edges)')

In [ ]:
# --- Top 15 most connected artists (degree centrality) ---
degree = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:15]
artists_top, degrees_top = zip(*degree)

plt.figure(figsize=(12, 5))
plt.barh(list(reversed(artists_top)), list(reversed(degrees_top)), color='steelblue')
plt.xlabel('Number of Collaborations')
plt.title('Top 15 Most Collaborative Artists', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Recommender Engine

Given a song title, compute Cosine Similarity across the combined feature matrix and return Top-N similar tracks.

In [ ]:
def recommend(song_name: str, df, X, top_n: int = 10):
    """
    Content-based recommender using Cosine Similarity on the combined sparse feature matrix.

    Improvements over naive version:
    1. Averages the feature vector across ALL duplicate rows for the query song
       (a song may appear many times in the dataset with slightly different metadata).
    2. Excludes every row that shares the exact same (track_name, artists) pair as the
       query, not just the first matching row.
    3. Deduplicates the result list by (track_name, artists) so the same song is never
       recommended twice, even when it exists in multiple genres/entries.
    """
    import scipy.sparse as sp
    from sklearn.metrics.pairwise import cosine_similarity

    # --- 1. Locate all rows for the requested song ---
    mask_query = df['track_name'].str.lower() == song_name.lower()
    matches = df[mask_query]
    if matches.empty:
        print(f'Song "{song_name}" not found in dataset.')
        return

    # Use the (track_name, artists) of the highest-popularity entry as canonical identity
    canonical = matches.loc[matches['popularity'].idxmax()]
    query_track = canonical['track_name']
    query_artist = canonical['artists']
    print(f'Finding recommendations for: {query_track} — {query_artist}')

    # --- 2. Build query vector as the mean of all matching rows ---
    match_indices = matches.index.tolist()
    song_vector = X[match_indices].mean(axis=0)          # dense (1 x n_features)
    song_vector_sparse = sp.csr_matrix(song_vector)      # back to sparse for cosine_similarity

    # --- 3. Compute cosine similarity against every track ---
    scores = cosine_similarity(song_vector_sparse, X).flatten()

    # --- 4. Mask out ALL rows that share (track_name, artists) with the query ---
    exclude_mask = (
        (df['track_name'].str.lower() == query_track.lower()) &
        (df['artists'].str.lower() == query_artist.lower())
    )
    scores[exclude_mask] = -1  # force them below any real candidate

    # --- 5. Rank all remaining tracks ---
    ranked_indices = scores.argsort()[::-1]

    results = df.loc[ranked_indices, ['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity'] = scores[ranked_indices].round(4)

    # --- 6. Deduplicate by (track_name, artists) — keep highest similarity per unique song ---
    results['_key'] = results['track_name'].str.lower() + '|||' + results['artists'].str.lower()
    results = results.drop_duplicates(subset='_key', keep='first')
    results = results.drop(columns='_key')

    # Remove excluded rows (safety net) and return top-N
    results = results[results['similarity'] >= 0].head(top_n)
    return results.reset_index(drop=True)


In [ ]:
# --- Test the recommender ---
# Example 1: latin track that previously returned duplicates of itself
recommend('La cancion', df, X_combined, top_n=10)


In [ ]:
# Example 2: classic rock track
recommend('Bohemian Rhapsody', df, X_combined, top_n=10)
